读取文件

In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data/raw/prices_2016-01-01_2026-01-01.csv").is_file()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Cannot find the project root.")

XGB_FILE = ROOT / "reports" / "xgboost_predictions.csv"
RF_FILE = ROOT / "reports" / "random_forest_predictions.csv"
PRICES_FILE = ROOT / "data/raw/prices_2016-01-01_2026-01-01.csv"

xgb_predictions = pd.read_csv(
    XGB_FILE,
    parse_dates=["Date"],
)

rf_predictions = pd.read_csv(
    RF_FILE,
    parse_dates=["Date"],
)

raw_prices = pd.read_csv(
    PRICES_FILE,
    header=[0, 1],
    index_col=0,
    parse_dates=[0],
    float_precision="round_trip",
)

print("XGBoost prediction rows:", len(xgb_predictions))
print("Random Forest prediction rows:", len(rf_predictions))
print("Raw price shape:", raw_prices.shape)
print("Price date range:", raw_prices.index.min(), "to", raw_prices.index.max())

display(xgb_predictions.head())
display(rf_predictions.head())

XGBoost prediction rows: 249133
Random Forest prediction rows: 249133
Raw price shape: (2514, 3018)
Price date range: 2016-01-04 00:00:00 to 2025-12-31 00:00:00


,Date,Ticker,prediction
0,2024-01-02,A,0.000454
1,2024-01-02,AAPL,-0.000755
2,2024-01-02,ABBV,0.000184
3,2024-01-02,ABNB,0.000666
4,2024-01-02,ABT,0.000520


,Date,Ticker,prediction
0,2024-01-02,A,0.000510
1,2024-01-02,AAPL,0.000282
2,2024-01-02,ABBV,0.000485
3,2024-01-02,ABNB,0.000520
4,2024-01-02,ABT,0.000510


整理价格（每行：某只股票在某天的开盘和收盘价）

In [7]:
quotes = (
    raw_prices
    .stack(level="Ticker", future_stack=True)
    .rename_axis(["Date", "Ticker"])
    .reset_index()
)

quotes.columns.name = None

quotes = quotes[[
    "Date",
    "Ticker",
    "Open",
    "Close",
]]

quotes = quotes.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

print("Quote rows:", len(quotes))
print("Unique tickers:", quotes["Ticker"].nunique())
print("Quote date range:", quotes["Date"].min(), "to", quotes["Date"].max())

display(quotes.head())

Quote rows: 1264542
Unique tickers: 503
Quote date range: 2016-01-04 00:00:00 to 2025-12-31 00:00:00


,Date,Ticker,Open,Close
0,2016-01-04,A,41.060001,40.689999
1,2016-01-05,A,40.730000,40.549999
2,2016-01-06,A,40.240002,40.730000
3,2016-01-07,A,40.139999,39.000000
4,2016-01-08,A,39.220001,38.590000


In [8]:
quotes["next_date"] = (
    quotes.groupby("Ticker")["Date"].shift(-1)
)

quotes["next_open"] = (
    quotes.groupby("Ticker")["Open"].shift(-1)
)

quotes["next_close"] = (
    quotes.groupby("Ticker")["Close"].shift(-1)
)

trade_quotes = quotes[[
    "Date",
    "Ticker",
    "next_date",
    "next_open",
    "next_close",
]].copy()

print("Trade quote rows:", len(trade_quotes))
print("Rows with next open:", trade_quotes["next_open"].notna().sum())
print("Rows with next close:", trade_quotes["next_close"].notna().sum())

display(trade_quotes.head())

Trade quote rows: 1264542
Rows with next open: 1217351
Rows with next close: 1217351


,Date,Ticker,next_date,next_open,next_close
0,2016-01-04,A,2016-01-05,40.730000,40.549999
1,2016-01-05,A,2016-01-06,40.240002,40.730000
2,2016-01-06,A,2016-01-07,40.139999,39.000000
3,2016-01-07,A,2016-01-08,39.220001,38.590000
4,2016-01-08,A,2016-01-11,38.709999,37.939999


把 XGBoost 预测和实际交易价格对应起来

In [9]:
xgb_candidates = xgb_predictions.merge(
    trade_quotes,
    on=["Date", "Ticker"],
    how="left",
    validate="one_to_one",
)

xgb_candidates = xgb_candidates.sort_values(
    ["Date", "prediction", "Ticker"],
    ascending=[True, False, True],
).reset_index(drop=True)

xgb_candidates["prediction_rank"] = (
    xgb_candidates.groupby("Date").cumcount() + 1
)

xgb_top10 = xgb_candidates.loc[
    xgb_candidates["prediction_rank"] <= 10
].copy()

print("XGBoost candidate rows:", len(xgb_candidates))
print("XGBoost selected rows:", len(xgb_top10))
print(
    "Selected rows with missing next open:",
    xgb_top10["next_open"].isna().sum(),
)
print(
    "Selected rows with missing next close:",
    xgb_top10["next_close"].isna().sum(),
)

first_test_date = xgb_top10["Date"].min()
display(
    xgb_top10.loc[
        xgb_top10["Date"].eq(first_test_date),
        [
            "Date",
            "Ticker",
            "prediction",
            "prediction_rank",
            "next_date",
            "next_open",
            "next_close",
        ],
    ]
)

XGBoost candidate rows: 249133
XGBoost selected rows: 5020
Selected rows with missing next open: 10
Selected rows with missing next close: 10


,Date,Ticker,prediction,prediction_rank,next_date,next_open,next_close
0,2024-01-02,NCLH,0.018089,1,2024-01-03,17.760000,17.320000
1,2024-01-02,COIN,0.016038,2,2024-01-03,146.000000,152.240005
2,2024-01-02,XYZ,0.009189,3,2024-01-03,69.000000,68.629997
3,2024-01-02,RCL,0.009004,4,2024-01-03,117.599998,117.449997
4,2024-01-02,MRNA,0.008611,5,2024-01-03,112.510002,108.580002
5,2024-01-02,CCL,0.007845,6,2024-01-03,16.760000,16.350000
6,2024-01-02,CVNA,0.006955,7,2024-01-03,9.300000,9.338000
7,2024-01-02,SKYD,0.002004,8,2024-01-03,14.130000,14.190000
8,2024-01-02,BE,0.001813,9,2024-01-03,14.360000,14.020000
9,2024-01-02,SMCI,0.001800,10,2024-01-03,27.725000,28.063999


计算XGBoost的实际交易收益

In [10]:
TRANSACTION_COST = 0.001
PORTFOLIO_SLOTS = 10

xgb_trades = xgb_top10.copy()

has_open = xgb_trades["next_open"].notna()
has_close = xgb_trades["next_close"].notna()

unresolved = has_open & ~has_close

print("Selected rows without next open:", (~has_open).sum())
print("Bought rows without next close:", unresolved.sum())

if unresolved.any():
    raise ValueError(
        "Some selected stocks have an entry price but no exit price."
    )

xgb_trades["gross_return"] = np.nan
xgb_trades["net_return"] = 0.0

complete_trade = has_open & has_close

xgb_trades.loc[complete_trade, "gross_return"] = (
    xgb_trades.loc[complete_trade, "next_close"]
    / xgb_trades.loc[complete_trade, "next_open"]
    - 1
)

xgb_trades.loc[complete_trade, "net_return"] = (
    (1 + xgb_trades.loc[complete_trade, "gross_return"])
    * (1 - TRANSACTION_COST) ** 2
    - 1
)

xgb_daily_returns = (
    xgb_trades
    .groupby("Date")["net_return"]
    .sum()
    / PORTFOLIO_SLOTS
)

print("Backtest dates:", len(xgb_daily_returns))
print("Mean daily net return:", xgb_daily_returns.mean())
print("Cumulative net return:", (1 + xgb_daily_returns).prod() - 1)

display(xgb_trades.head())
display(xgb_daily_returns.head())

Selected rows without next open: 10
Bought rows without next close: 0
Backtest dates: 502
Mean daily net return: -0.0011883741926845596
Cumulative net return: -0.48958364470544713


,Date,Ticker,prediction,next_date,next_open,next_close,prediction_rank,gross_return,net_return
0,2024-01-02,NCLH,0.018089,2024-01-03,17.760000,17.320000,1,-0.024775,-0.026724
1,2024-01-02,COIN,0.016038,2024-01-03,146.000000,152.240005,2,0.042740,0.040655
2,2024-01-02,XYZ,0.009189,2024-01-03,69.000000,68.629997,3,-0.005362,-0.007351
3,2024-01-02,RCL,0.009004,2024-01-03,117.599998,117.449997,4,-0.001276,-0.003272
4,2024-01-02,MRNA,0.008611,2024-01-03,112.510002,108.580002,5,-0.034930,-0.036859


Date
2024-01-02   -0.007107
2024-01-03    0.003943
2024-01-04    0.001656
2024-01-05    0.022273
2024-01-08    0.003517
Name: net_return, dtype: float64

计算 XGBoost 的交易前后收益、年化 Sharpe 和最大回撤

In [11]:
xgb_daily_gross = (
    xgb_trades
    .assign(gross_return=xgb_trades["gross_return"].fillna(0.0))
    .groupby("Date")["gross_return"]
    .sum()
    / PORTFOLIO_SLOTS
)

xgb_equity_net = (1 + xgb_daily_returns).cumprod()
xgb_previous_high = xgb_equity_net.cummax()
xgb_drawdown = xgb_equity_net / xgb_previous_high - 1

xgb_summary = {
    "Mean daily gross return": xgb_daily_gross.mean(),
    "Mean daily net return": xgb_daily_returns.mean(),
    "Cumulative gross return": (1 + xgb_daily_gross).prod() - 1,
    "Cumulative net return": (1 + xgb_daily_returns).prod() - 1,
    "Annualized daily Sharpe": (
        xgb_daily_returns.mean()
        / xgb_daily_returns.std()
        * np.sqrt(252)
    ),
    "Maximum drawdown": xgb_drawdown.min(),
}

display(pd.Series(xgb_summary, name="XGBoost"))

Mean daily gross return    0.000808
Mean daily net return     -0.001188
Cumulative gross return    0.390930
Cumulative net return     -0.489584
Annualized daily Sharpe   -1.083708
Maximum drawdown          -0.522173
Name: XGBoost, dtype: float64

XGBoost 的预测信号有一点点毛收益，但信号太弱，无法覆盖交易成本；按项目规定的交易规则，最终净收益很差

完全相同的回测规则测试随机森林

In [12]:
def run_top10_backtest(predictions, model_name):
    candidates = predictions.merge(
        trade_quotes,
        on=["Date", "Ticker"],
        how="left",
        validate="one_to_one",
    )

    candidates = candidates.sort_values(
        ["Date", "prediction", "Ticker"],
        ascending=[True, False, True],
    ).reset_index(drop=True)

    candidates["prediction_rank"] = (
        candidates.groupby("Date").cumcount() + 1
    )

    selected = candidates.loc[
        candidates["prediction_rank"] <= PORTFOLIO_SLOTS
    ].copy()

    has_open = selected["next_open"].notna()
    has_close = selected["next_close"].notna()
    unresolved = has_open & ~has_close

    if unresolved.any():
        raise ValueError(
            f"{model_name}: selected trades are missing exit prices."
        )

    selected["gross_return"] = np.nan
    selected["net_return"] = 0.0

    complete_trade = has_open & has_close

    selected.loc[complete_trade, "gross_return"] = (
        selected.loc[complete_trade, "next_close"]
        / selected.loc[complete_trade, "next_open"]
        - 1
    )

    selected.loc[complete_trade, "net_return"] = (
        (1 + selected.loc[complete_trade, "gross_return"])
        * (1 - TRANSACTION_COST) ** 2
        - 1
    )

    daily_gross = (
        selected
        .assign(gross_return=selected["gross_return"].fillna(0.0))
        .groupby("Date")["gross_return"]
        .sum()
        / PORTFOLIO_SLOTS
    )

    daily_net = (
        selected
        .groupby("Date")["net_return"]
        .sum()
        / PORTFOLIO_SLOTS
    )

    equity_net = (1 + daily_net).cumprod()
    previous_high = equity_net.cummax()
    drawdown = equity_net / previous_high - 1

    summary = pd.Series(
        {
            "Mean daily gross return": daily_gross.mean(),
            "Mean daily net return": daily_net.mean(),
            "Cumulative gross return": (1 + daily_gross).prod() - 1,
            "Cumulative net return": (1 + daily_net).prod() - 1,
            "Annualized daily Sharpe": (
                daily_net.mean()
                / daily_net.std()
                * np.sqrt(252)
            ),
            "Maximum drawdown": drawdown.min(),
            "Missing next open rows": (~has_open).sum(),
            "Backtest dates": len(daily_net),
        },
        name=model_name,
    )

    return selected, daily_net, summary


rf_trades, rf_daily_returns, rf_summary = run_top10_backtest(
    rf_predictions,
    "Random Forest",
)

display(rf_summary)

Mean daily gross return      0.001094
Mean daily net return       -0.000903
Cumulative gross return      0.605105
Cumulative net return       -0.410990
Annualized daily Sharpe     -0.820478
Maximum drawdown            -0.487302
Missing next open rows      10.000000
Backtest dates             502.000000
Name: Random Forest, dtype: float64

baseline

In [13]:
benchmark = xgb_predictions[["Date", "Ticker"]].merge(
    trade_quotes,
    on=["Date", "Ticker"],
    how="left",
    validate="one_to_one",
)

has_open = benchmark["next_open"].notna()
has_close = benchmark["next_close"].notna()
unresolved = has_open & ~has_close

if unresolved.any():
    raise ValueError(
        "Benchmark has bought rows without exit prices."
    )

benchmark["gross_return"] = 0.0
benchmark["net_return"] = 0.0

complete_trade = has_open & has_close

benchmark.loc[complete_trade, "gross_return"] = (
    benchmark.loc[complete_trade, "next_close"]
    / benchmark.loc[complete_trade, "next_open"]
    - 1
)

benchmark.loc[complete_trade, "net_return"] = (
    (1 + benchmark.loc[complete_trade, "gross_return"])
    * (1 - TRANSACTION_COST) ** 2
    - 1
)

eligible_counts = benchmark.groupby("Date")["Ticker"].size()

benchmark_daily_gross = (
    benchmark.groupby("Date")["gross_return"].sum()
    / eligible_counts
)

benchmark_daily_net = (
    benchmark.groupby("Date")["net_return"].sum()
    / eligible_counts
)

benchmark_equity = (1 + benchmark_daily_net).cumprod()
benchmark_drawdown = (
    benchmark_equity / benchmark_equity.cummax() - 1
)

benchmark_summary = pd.Series(
    {
        "Mean daily gross return": benchmark_daily_gross.mean(),
        "Mean daily net return": benchmark_daily_net.mean(),
        "Cumulative gross return": (
            1 + benchmark_daily_gross
        ).prod() - 1,
        "Cumulative net return": (
            1 + benchmark_daily_net
        ).prod() - 1,
        "Annualized daily Sharpe": (
            benchmark_daily_net.mean()
            / benchmark_daily_net.std()
            * np.sqrt(252)
        ),
        "Maximum drawdown": benchmark_drawdown.min(),
    },
    name="Equal-weight benchmark",
)

display(benchmark_summary)

Mean daily gross return    0.000193
Mean daily net return     -0.001803
Cumulative gross return    0.082418
Cumulative net return     -0.602796
Annualized daily Sharpe   -3.412913
Maximum drawdown          -0.603033
Name: Equal-weight benchmark, dtype: float64

In [15]:
xgb_summary_series = pd.Series(
    xgb_summary,
    name="XGBoost",
)

comparison = pd.concat(
    [
        xgb_summary_series,
        rf_summary,
        benchmark_summary,
    ],
    axis=1,
).T

comparison.index.name = "strategy"

display(comparison)

SUMMARY_FILE = ROOT / "reports" / "backtest_summary.csv"
comparison.to_csv(SUMMARY_FILE)

print("Saved:", SUMMARY_FILE)

,Mean daily gross return,Mean daily net return,Cumulative gross return,Cumulative net return,Annualized daily Sharpe,Maximum drawdown,Missing next open rows,Backtest dates
strategy,,,,,,,,
XGBoost,0.000808,-0.001188,0.390930,-0.489584,-1.083708,-0.522173,NaN,NaN
Random Forest,0.001094,-0.000903,0.605105,-0.410990,-0.820478,-0.487302,10.0,502.0
Equal-weight benchmark,0.000193,-0.001803,0.082418,-0.602796,-3.412913,-0.603033,NaN,NaN


Saved: /Users/chenxinhui/financial-engineering-club-fall26/reports/backtest_summary.csv


交易成本敏感性分析

In [16]:
def cost_sensitivity(trades, model_name):
    rows = []

    for cost in [0.0, 0.00025, 0.0005, 0.001]:
        trade_returns = pd.Series(
            0.0,
            index=trades.index,
        )

        complete_trade = trades["gross_return"].notna()

        trade_returns.loc[complete_trade] = (
            (1 + trades.loc[complete_trade, "gross_return"])
            * (1 - cost) ** 2
            - 1
        )

        daily_returns = (
            pd.DataFrame({
                "Date": trades["Date"],
                "return": trade_returns,
            })
            .groupby("Date")["return"]
            .sum()
            / PORTFOLIO_SLOTS
        )

        equity = (1 + daily_returns).cumprod()
        drawdown = equity / equity.cummax() - 1

        rows.append({
            "model": model_name,
            "cost_each_way": cost,
            "mean_daily_return": daily_returns.mean(),
            "cumulative_return": (1 + daily_returns).prod() - 1,
            "annualized_sharpe": (
                daily_returns.mean()
                / daily_returns.std()
                * np.sqrt(252)
            ),
            "maximum_drawdown": drawdown.min(),
        })

    return pd.DataFrame(rows)


cost_results = pd.concat(
    [
        cost_sensitivity(xgb_trades, "XGBoost"),
        cost_sensitivity(rf_trades, "Random Forest"),
    ],
    ignore_index=True,
)

display(cost_results)

,model,cost_each_way,mean_daily_return,cumulative_return,annualized_sharpe,maximum_drawdown
0,XGBoost,0.00000,0.000808,0.390930,0.735600,-0.266465
1,XGBoost,0.00025,0.000309,0.082682,0.281285,-0.315139
2,XGBoost,0.00050,-0.000190,-0.157307,-0.173373,-0.370735
3,XGBoost,0.00100,-0.001188,-0.489584,-1.083708,-0.522173
4,Random Forest,0.00000,0.001094,0.605105,0.991830,-0.249680
5,Random Forest,0.00025,0.000595,0.249393,0.539264,-0.299712
6,Random Forest,0.00050,0.000095,-0.027549,0.086357,-0.347724
7,Random Forest,0.00100,-0.000903,-0.410990,-0.820478,-0.487302


In [19]:
LAMBDA_FILE = ROOT / "reports" / "lambdamart_predictions.csv"

lambdamart_predictions = pd.read_csv(
    LAMBDA_FILE,
    parse_dates=["Date"],
)

print("LambdaMART prediction rows:", len(lambdamart_predictions))
display(lambdamart_predictions.head())

LambdaMART prediction rows: 249133


,Date,Ticker,prediction
0,2024-01-02,A,-0.237463
1,2024-01-02,AAPL,-0.135531
2,2024-01-02,ABBV,-0.092277
3,2024-01-02,ABNB,-0.230522
4,2024-01-02,ABT,-0.342587


In [20]:
lambdamart_trades, lambdamart_daily_returns, lambdamart_summary = (
    run_top10_backtest(
        lambdamart_predictions,
        "LambdaMART",
    )
)

display(lambdamart_summary)

Mean daily gross return      0.001187
Mean daily net return       -0.000810
Cumulative gross return      0.618356
Cumulative net return       -0.406127
Annualized daily Sharpe     -0.600864
Maximum drawdown            -0.481391
Missing next open rows      10.000000
Backtest dates             502.000000
Name: LambdaMART, dtype: float64

In [21]:
lambda_cost_results = cost_sensitivity(
    lambdamart_trades,
    "LambdaMART",
)

display(lambda_cost_results)

,model,cost_each_way,mean_daily_return,cumulative_return,annualized_sharpe,maximum_drawdown
0,LambdaMART,0.00000,0.001187,0.618356,0.879085,-0.254519
1,LambdaMART,0.00025,0.000688,0.259708,0.509515,-0.281242
2,LambdaMART,0.00050,0.000188,-0.019521,0.139667,-0.332773
3,LambdaMART,0.00100,-0.000810,-0.406127,-0.600864,-0.481391


## LambdaMART 持仓回测规则

- 在交易日收盘后，根据 LambdaMART 预测分数进行排名；
- 股票进入 Top 10 时产生买入信号；
- 买入后继续持有；
- 只有当股票跌出 Top 50 时才产生卖出信号；
- 买卖在下一交易日开盘执行；
- 不使用未来交易日的最高价、最低价或收盘价生成当前信号；
- 每次买入和卖出分别收取 0.10% 交易成本；
- 未成交或缺少有效价格的仓位保持现金。

In [35]:
ENTRY_RANK = 10
EXIT_RANK = 50

lambda_signals = lambdamart_predictions.merge(
    trade_quotes,
    on=["Date", "Ticker"],
    how="left",
    validate="one_to_one",
)

lambda_signals = lambda_signals.sort_values(
    ["Date", "prediction", "Ticker"],
    ascending=[True, False, True],
).reset_index(drop=True)

lambda_signals["prediction_rank"] = (
    lambda_signals.groupby("Date").cumcount() + 1
)

lambda_signals["entry_signal"] = (
    lambda_signals["prediction_rank"] <= ENTRY_RANK
)

lambda_signals["exit_signal"] = (
    lambda_signals["prediction_rank"] > EXIT_RANK
)

first_signal_date = lambda_signals["Date"].min()

display(
    lambda_signals.loc[
        lambda_signals["Date"].eq(first_signal_date),
        [
            "Date",
            "Ticker",
            "prediction",
            "prediction_rank",
            "entry_signal",
            "exit_signal",
        ],
    ].head(15)
)

print(
    "Entry signals on first date:",
    lambda_signals.loc[
        lambda_signals["Date"].eq(first_signal_date),
        "entry_signal",
    ].sum(),
)

,Date,Ticker,prediction,prediction_rank,entry_signal,exit_signal
0,2024-01-02,COIN,0.361704,1,True,False
1,2024-01-02,MRNA,0.345935,2,True,False
2,2024-01-02,XYZ,0.300403,3,True,False
3,2024-01-02,NCLH,0.245459,4,True,False
4,2024-01-02,ECHO,0.245230,5,True,False
5,2024-01-02,CVNA,0.244689,6,True,False
6,2024-01-02,RCL,0.198536,7,True,False
7,2024-01-02,BE,0.187090,8,True,False
8,2024-01-02,VTRS,0.172337,9,True,False
9,2024-01-02,CCL,0.149660,10,True,False


Entry signals on first date: 10


In [36]:
held = set()
holding_records = []
turnover_records = []

for date, day in lambda_signals.groupby("Date", sort=True):
    day = day.sort_values(
        ["prediction_rank", "Ticker"]
    )

    previous_held = held.copy()

    rank_by_ticker = (
        day.set_index("Ticker")["prediction_rank"]
    )

    # 继续持有尚未跌出 Top 50 的股票
    held = {
        ticker
        for ticker in previous_held
        if ticker in rank_by_ticker.index
        and rank_by_ticker.loc[ticker] <= EXIT_RANK
    }

    # 如果仓位不足 10 个，从新的 Top 10 股票中补充
    entry_candidates = day.loc[
        day["prediction_rank"] <= ENTRY_RANK,
        "Ticker",
    ].tolist()

    for ticker in entry_candidates:
        if len(held) >= ENTRY_RANK:
            break
        held.add(ticker)

    bought = held - previous_held
    sold = previous_held - held

    turnover_records.append({
        "Date": date,
        "held_count": len(held),
        "bought_count": len(bought),
        "sold_count": len(sold),
    })

    for ticker in sorted(held):
        holding_records.append({
            "Date": date,
            "Ticker": ticker,
            "prediction_rank": rank_by_ticker.loc[ticker],
            "is_new_position": ticker in bought,
        })

lambda_holdings = pd.DataFrame(holding_records)
lambda_turnover = pd.DataFrame(turnover_records)

print("Holding rows:", len(lambda_holdings))
print("Average held positions:", lambda_turnover["held_count"].mean())
print("Total buys:", lambda_turnover["bought_count"].sum())
print("Total sells:", lambda_turnover["sold_count"].sum())

display(lambda_turnover.head())
display(lambda_holdings.head(10))

Holding rows: 5020
Average held positions: 10.0
Total buys: 498
Total sells: 488


,Date,held_count,bought_count,sold_count
0,2024-01-02,10,10,0
1,2024-01-03,10,2,2
2,2024-01-04,10,1,1
3,2024-01-05,10,1,1
4,2024-01-08,10,0,0


,Date,Ticker,prediction_rank,is_new_position
0,2024-01-02,BE,8,True
1,2024-01-02,CCL,10,True
2,2024-01-02,COIN,1,True
3,2024-01-02,CVNA,6,True
4,2024-01-02,ECHO,5,True
5,2024-01-02,MRNA,2,True
6,2024-01-02,NCLH,4,True
7,2024-01-02,RCL,7,True
8,2024-01-02,VTRS,9,True
9,2024-01-02,XYZ,3,True


回测

In [37]:
quote_lookup = (
    quotes
    .set_index(["Date", "Ticker"])[["Open", "Close"]]
)

desired_positions = lambda_holdings.merge(
    trade_quotes[["Date", "Ticker", "next_date"]],
    on=["Date", "Ticker"],
    how="left",
    validate="one_to_one",
).dropna(subset=["next_date"])

desired_by_execution_date = {
    execution_date: set(group["Ticker"])
    for execution_date, group in desired_positions.groupby("next_date")
}

def get_quote(date, ticker):
    try:
        return quote_lookup.loc[(date, ticker)]
    except KeyError:
        return None

cash = 1.0
shares = {}
previous_close_value = 1.0
portfolio_records = []

for execution_date in sorted(desired_by_execution_date):
    desired = desired_by_execution_date[execution_date]

    # 先卖出已经跌出 Top 50 的股票
    sold_count = 0

    for ticker in list(shares):
        quote = get_quote(execution_date, ticker)

        if quote is None or pd.isna(quote["Open"]):
            raise ValueError(
                f"Cannot value or sell held position: "
                f"{execution_date}, {ticker}"
            )

        if ticker not in desired:
            sale_value = (
                shares[ticker]
                * quote["Open"]
                * (1 - TRANSACTION_COST)
            )
            cash += sale_value
            del shares[ticker]
            sold_count += 1

    # 计算卖出后的组合价值
    open_value = cash

    for ticker, quantity in shares.items():
        quote = get_quote(execution_date, ticker)

        if quote is None or pd.isna(quote["Open"]):
            raise ValueError(
                f"Cannot value held position: "
                f"{execution_date}, {ticker}"
            )

        open_value += quantity * quote["Open"]

    # 买入新进入 Top 10 的股票
    bought_count = 0
    new_positions = desired - set(shares)

    for ticker in sorted(new_positions):
        quote = get_quote(execution_date, ticker)

        if quote is None or pd.isna(quote["Open"]):
            # 没有有效开盘价，保留现金，不替换股票
            continue

        allocation = min(
            open_value / ENTRY_RANK,
            cash / (1 + TRANSACTION_COST),
        )

        if allocation <= 0:
            continue

        quantity = allocation / (
            quote["Open"] * (1 + TRANSACTION_COST)
        )

        cash -= quantity * quote["Open"] * (
            1 + TRANSACTION_COST
        )

        shares[ticker] = quantity
        bought_count += 1

    # 收盘时估值
    close_value = cash

    for ticker, quantity in shares.items():
        quote = get_quote(execution_date, ticker)

        if quote is None or pd.isna(quote["Close"]):
            raise ValueError(
                f"Bought position has no closing price: "
                f"{execution_date}, {ticker}"
            )

        close_value += quantity * quote["Close"]

    daily_return = close_value / previous_close_value - 1

    portfolio_records.append({
        "Date": execution_date,
        "portfolio_value": close_value,
        "daily_return": daily_return,
        "bought_count": bought_count,
        "sold_count": sold_count,
        "held_count": len(shares),
        "cash": cash,
    })

    previous_close_value = close_value

lambda_portfolio = pd.DataFrame(portfolio_records)

print("Portfolio dates:", len(lambda_portfolio))
print("Total buys:", lambda_portfolio["bought_count"].sum())
print("Total sells:", lambda_portfolio["sold_count"].sum())
print(
    "Cumulative net return:",
    lambda_portfolio["portfolio_value"].iloc[-1] - 1,
)

display(lambda_portfolio.head())

Portfolio dates: 501
Total buys: 498
Total sells: 488
Cumulative net return: 2.808810989754151


,Date,portfolio_value,daily_return,bought_count,sold_count,held_count,cash
0,2024-01-03,0.984022,-0.015978,10,0,10,0.000100
1,2024-01-04,0.983319,-0.000714,2,2,10,0.003214
2,2024-01-05,0.982004,-0.001337,1,1,10,0.004574
3,2024-01-08,1.006624,0.025071,1,1,10,0.002031
4,2024-01-09,0.985874,-0.020613,0,0,10,0.002031


In [38]:
portfolio_returns = lambda_portfolio["daily_return"]

portfolio_equity = lambda_portfolio["portfolio_value"]
portfolio_drawdown = (
    portfolio_equity / portfolio_equity.cummax() - 1
)

lambda_hold_summary = pd.Series({
    "Mean daily return": portfolio_returns.mean(),
    "Cumulative net return": portfolio_equity.iloc[-1] - 1,
    "Annualized daily Sharpe": (
        portfolio_returns.mean()
        / portfolio_returns.std()
        * np.sqrt(252)
    ),
    "Maximum drawdown": portfolio_drawdown.min(),
    "Total buys": lambda_portfolio["bought_count"].sum(),
    "Total sells": lambda_portfolio["sold_count"].sum(),
    "Average held positions": lambda_portfolio["held_count"].mean(),
}, name="LambdaMART Top-10 / Top-50 hold")

display(lambda_hold_summary)

Mean daily return            0.003095
Cumulative net return        2.808811
Annualized daily Sharpe      1.687769
Maximum drawdown            -0.400444
Total buys                 498.000000
Total sells                488.000000
Average held positions      10.000000
Name: LambdaMART Top-10 / Top-50 hold, dtype: float64

In [39]:
final_comparison = pd.concat(
    [
        lambdamart_summary,
        lambda_hold_summary,
        benchmark_summary,
    ],
    axis=1,
).T

final_comparison.index.name = "strategy"

display(final_comparison)

FINAL_SUMMARY_FILE = (
    ROOT / "reports" / "final_backtest_summary.csv"
)

final_comparison.to_csv(FINAL_SUMMARY_FILE)

print("Saved:", FINAL_SUMMARY_FILE)

,Mean daily gross return,Mean daily net return,Cumulative gross return,Cumulative net return,Annualized daily Sharpe,Maximum drawdown,Missing next open rows,Backtest dates,Mean daily return,Total buys,Total sells,Average held positions
strategy,,,,,,,,,,,,
LambdaMART,0.001187,-0.000810,0.618356,-0.406127,-0.600864,-0.481391,10.0,502.0,NaN,NaN,NaN,NaN
LambdaMART Top-10 / Top-50 hold,NaN,NaN,NaN,2.808811,1.687769,-0.400444,NaN,NaN,0.003095,498.0,488.0,10.0
Equal-weight benchmark,0.000193,-0.001803,0.082418,-0.602796,-3.412913,-0.603033,NaN,NaN,NaN,NaN,NaN,NaN


Saved: /Users/chenxinhui/financial-engineering-club-fall26/reports/final_backtest_summary.csv


In [40]:
clean_summary = pd.DataFrame([
    {
        "strategy": "LambdaMART daily Top-10",
        "mean_daily_net_return": lambdamart_summary[
            "Mean daily net return"
        ],
        "cumulative_net_return": lambdamart_summary[
            "Cumulative net return"
        ],
        "annualized_sharpe": lambdamart_summary[
            "Annualized daily Sharpe"
        ],
        "maximum_drawdown": lambdamart_summary[
            "Maximum drawdown"
        ],
    },
    {
        "strategy": "LambdaMART Top-10 / Top-50 hold",
        "mean_daily_net_return": lambda_hold_summary[
            "Mean daily return"
        ],
        "cumulative_net_return": lambda_hold_summary[
            "Cumulative net return"
        ],
        "annualized_sharpe": lambda_hold_summary[
            "Annualized daily Sharpe"
        ],
        "maximum_drawdown": lambda_hold_summary[
            "Maximum drawdown"
        ],
    },
    {
        "strategy": "Equal-weight benchmark",
        "mean_daily_net_return": benchmark_summary[
            "Mean daily net return"
        ],
        "cumulative_net_return": benchmark_summary[
            "Cumulative net return"
        ],
        "annualized_sharpe": benchmark_summary[
            "Annualized daily Sharpe"
        ],
        "maximum_drawdown": benchmark_summary[
            "Maximum drawdown"
        ],
    },
])

display(clean_summary)

CLEAN_SUMMARY_FILE = (
    ROOT / "reports" / "final_backtest_summary_clean.csv"
)

clean_summary.to_csv(CLEAN_SUMMARY_FILE, index=False)

print("Saved:", CLEAN_SUMMARY_FILE)

,strategy,mean_daily_net_return,cumulative_net_return,annualized_sharpe,maximum_drawdown
0,LambdaMART daily Top-10,-0.000810,-0.406127,-0.600864,-0.481391
1,LambdaMART Top-10 / Top-50 hold,0.003095,2.808811,1.687769,-0.400444
2,Equal-weight benchmark,-0.001803,-0.602796,-3.412913,-0.603033


Saved: /Users/chenxinhui/financial-engineering-club-fall26/reports/final_backtest_summary_clean.csv
